# 07.03 | Conditional continuous HRV profiles and synthesis

**Project:** Heart rate variability in older adults from southern Chile  
**Phase:** 07 — Two-factor HRV profiling  
**Notebook:** 03 — Stratified continuous associations, descriptive heterogeneity and cross-stage synthesis

## Research objective

Describe **continuous-factor versus cardiac-measure associations within pre-existing demographic, nutritional and blood-pressure strata**, then integrate their coverage with the fully descriptive joint-cell atlas (07.01) and the explicitly planned categorical contrasts (07.02).

The notebook is designed to exploit the continuous measurements **without dichotomizing them** or repeating the tests in notebooks 06 or 07.02. It estimates conditional rank associations (Spearman ρ and paired-bootstrap 95% CIs) within a predefined set of scientifically interpretable factor-stratifier pairs. For pairs with a *binary* stratifier, a direct **difference of two stratum-specific correlations** and a stratified bootstrap 95% CI provide an exploratory estimate of heterogeneity. This does **not** amount to a covariate-adjusted regression interaction test; differences of p-values alone are never interpreted as interaction evidence.

### Analysis boundaries

- All accepted source records remain in the cohort; original values, identities, categories and cutoffs remain unchanged. Each analysis uses the records belonging to its existing stratum and valid observed x/y pairs; no whole-cohort filtering, new exclusions, imputation or category fusion.
- Five main HRV measures (SDNN, RMSSD, HF, LF, VLF) are separated from two complementary cardiac measures (Mean RR, Mean HR).
- Spectral `log1p` is **visual only**. All inferential estimates use the untransformed source data.
- The 07.01 categorical support and 07.02 contrast results are **read from their saved CSVs**, never recomputed. Source 06.04 unifactorial Spearman estimates are an optional, explicitly labeled descriptive reference.
- Small, constant or empty strata are represented transparently. An undefined correlation gets no invented estimate, confidence interval or p/q-value.
- No causal inference, clinical diagnosis, study-wide FDR, or general factorial/interactions models are introduced.

## 0. Configuration and prespecified analysis plan

Run after **07.01** and **07.02**, from the project root (or a descendant). `INPUT_FILE_OVERRIDE` and `ANALYSIS_ROOT_OVERRIDE` permit explicit paths when needed. The expected accepted source is `outputs/dataset_final.csv`; all outputs stay in the dedicated 07.03 directory. Default 1,500 bootstrap draws and seed 20261128 maintain the approach of stage 06–07.02. Reduce `N_BOOT` only when testing execution, never for a final scientific run.

Seventeen factor–stratifier families cover eight continuous predictors. The families follow physiological interpretability and the available support; **they were defined without selecting on HRV p-values**.

In [1]:
from __future__ import annotations

from pathlib import Path
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import display

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    LOWESS_AVAILABLE = True
except ImportError:
    LOWESS_AVAILABLE = False

INPUT_FILE_OVERRIDE = None
ANALYSIS_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
N_BOOT = 1500
SEED = 20261128
CONFIDENCE_LEVEL = 0.95
PLOT_DPI = 200
SHOW_FIGURES = False

PRIMARY = {
    "t2m_pre_sdnn":"SDNN", "t2m_pre_rmssd":"RMSSD", "t2m_pre_hf":"HF",
    "t2m_pre_lf":"LF", "t2m_pre_vlf":"VLF",
}
COMPLEMENTARY = {"t2m_pre_mean_rr":"Mean RR", "t2m_pre_mean_hr":"Mean HR"}
MEASURES = {**PRIMARY, **COMPLEMENTARY}
OUTCOME_KEYS = list(MEASURES)

PREDICTOR_LABELS = {
    "age":"Age (years)", "weight_kg":"Weight (kg)", "height_cm":"Height (cm)",
    "imc":"BMI (kg/m²)", "bp_systolic":"SBP (mmHg)",
    "bp_diastolic":"DBP (mmHg)", "bp_pam":"MAP (mmHg)",
    "bp_pp":"Pulse pressure (mmHg)",
}
FACTOR_LEVELS = {
    "sex_category": ["Female", "Male"],
    "sbp_category": ["<140 mmHg", ">=140 mmHg"],
    "dbp_category": ["<90 mmHg", ">=90 mmHg"],
    "age_group": ["60-69", "70-79", "80+"],
}
STRATIFIER_LABELS = {
    "sex_category":"Sex", "sbp_category":"SBP category",
    "dbp_category":"DBP category", "age_group":"Age group",
}

# Analysis families: factor (continuous) × modifier (original categories).
# No HRV-outcome-driven selection and no new categorical cutoffs.
PAIR_SPECS = [
    ("age", "sex_category"),
    ("weight_kg", "sex_category"),
    ("height_cm", "sex_category"),
    ("imc", "sex_category"),
    ("bp_systolic", "sex_category"),
    ("bp_diastolic", "sex_category"),
    ("bp_pam", "sex_category"),
    ("bp_pp", "sex_category"),
    ("age", "sbp_category"),
    ("age", "dbp_category"),
    ("imc", "sbp_category"),
    ("imc", "dbp_category"),
    ("weight_kg", "age_group"),
    ("height_cm", "age_group"),
    ("imc", "age_group"),
    ("bp_systolic", "age_group"),
    ("bp_diastolic", "age_group"),
]
PLAN = pd.DataFrame([
    {"pair_id":f"P{i:02d}", "predictor":x,"stratifier":g,
     "predictor_label":PREDICTOR_LABELS[x], "stratifier_label":STRATIFIER_LABELS[g],
     "n_strata_planned":len(FACTOR_LEVELS[g]),
     "comparison_type":"descriptive_delta_rho" if len(FACTOR_LEVELS[g])==2 else "stratified_only"}
    for i,(x,g) in enumerate(PAIR_SPECS,1)
])

NAVY, MUTED, TEAL, CORAL, BLUE, GOLD = (
    "#18394C", "#627989", "#247E90", "#CC7255", "#5387AB", "#A48645"
)
STRATUM_COLORS = [TEAL,CORAL,BLUE,GOLD]
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({
    "figure.facecolor":"white","savefig.facecolor":"white","axes.facecolor":"white",
    "axes.edgecolor":"#DCE5EC","axes.labelcolor":NAVY,"text.color":NAVY,
    "xtick.color":MUTED,"ytick.color":MUTED,"grid.color":"#E6EDF1",
    "grid.alpha":.8,"axes.spines.top":False,"axes.spines.right":False,
    "figure.constrained_layout.use":False,"axes.titleweight":"semibold",
})
print("Planned continuous × categorical families:",len(PLAN))
print("Planned strata:", int(PLAN.n_strata_planned.sum()))
print("Bootstrap draws:",N_BOOT,"| LOWESS:",LOWESS_AVAILABLE)
display(PLAN)

Planned continuous × categorical families: 17
Planned strata: 39
Bootstrap draws: 1500 | LOWESS: True


,pair_id,predictor,stratifier,predictor_label,stratifier_label,n_strata_planned,comparison_type
0,P01,age,sex_category,Age (years),Sex,2,descriptive_delta_rho
1,P02,weight_kg,sex_category,Weight (kg),Sex,2,descriptive_delta_rho
2,P03,height_cm,sex_category,Height (cm),Sex,2,descriptive_delta_rho
3,P04,imc,sex_category,BMI (kg/m²),Sex,2,descriptive_delta_rho
4,P05,bp_systolic,sex_category,SBP (mmHg),Sex,2,descriptive_delta_rho
5,P06,bp_diastolic,sex_category,DBP (mmHg),Sex,2,descriptive_delta_rho
6,P07,bp_pam,sex_category,MAP (mmHg),Sex,2,descriptive_delta_rho
7,P08,bp_pp,sex_category,Pulse pressure (mmHg),Sex,2,descriptive_delta_rho
8,P09,age,sbp_category,Age (years),SBP category,2,descriptive_delta_rho
9,P10,age,dbp_category,Age (years),DBP category,2,descriptive_delta_rho


## 1. Load the accepted dataset and upstream stage 07 products

The accepted analytical CSV is read without modification. The stage 07.01 and 07.02 CSVs supply the previous analyses' **actual** coverage and estimates; they are not replaced with summaries or re-estimated here. The optional 06.04 continuous summary provides a descriptive reference only when available.

In [2]:
def find_project_input():
    if INPUT_FILE_OVERRIDE is not None:
        return Path(INPUT_FILE_OVERRIDE).expanduser().resolve()
    current=Path.cwd().resolve()
    for folder in (current,*current.parents):
        candidate=folder/"outputs"/"dataset_final.csv"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("outputs/dataset_final.csv not found; set INPUT_FILE_OVERRIDE.")

input_file=find_project_input()
project_root=(input_file.parent.parent if input_file.parent.name=="outputs" else input_file.parent)
analysis_root=(Path(ANALYSIS_ROOT_OVERRIDE).expanduser().resolve() if ANALYSIS_ROOT_OVERRIDE
               else project_root/"eda_results"/"hrv_associations")
output_root=(Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE
             else analysis_root/"07_03_conditional_continuous_profiles_and_synthesis")
tables_dir=output_root/"tables"
figure_dir=output_root/"figures"/"multipanels"
tables_dir.mkdir(parents=True,exist_ok=True)
figure_dir.mkdir(parents=True,exist_ok=True)

data=pd.read_csv(input_file)
required=[*PREDICTOR_LABELS,*FACTOR_LEVELS,*MEASURES]
missing=sorted(set(required)-set(data.columns))
if missing:
    raise KeyError(f"Accepted CSV lacks required analysis columns: {missing}")

stage01=analysis_root/"07_01_joint_categorical_hrv_profiles"/"tables"
stage02=analysis_root/"07_02_targeted_two_factor_hrv_comparisons"/"tables"
source_files={
    "cell_counts": stage01/"categorical_pairwise_cell_counts.csv",
    "cell_support": stage01/"categorical_pairwise_support_summary.csv",
    "joint_profiles": stage01/"joint_hrv_descriptive_profiles.csv",
    "planned_contrasts": stage02/"planned_two_factor_contrasts.csv",
    "targeted_effects": stage02/"targeted_two_factor_effects.csv",
    "targeted_families": stage02/"two_factor_fdr_family_summary.csv",
}
source_tables={}
for name,path in source_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"Required upstream stage 07 table not found: {path}. Run 07.01 and 07.02 first.")
    source_tables[name]=pd.read_csv(path)

unifactorial_path=analysis_root/"06_04_association_synthesis"/"tables"/"continuous_associations_combined.csv"
unifactorial_reference=(pd.read_csv(unifactorial_path) if unifactorial_path.is_file() else None)

print("Source:",input_file)
print("Input records (unchanged):",len(data))
print("Output:",output_root)
print("07.01 support rows:",len(source_tables["cell_support"]),"| joint profiles:",len(source_tables["joint_profiles"]))
print("07.02 planned contrasts:",len(source_tables["planned_contrasts"]),"| effects:",len(source_tables["targeted_effects"]))
print("06.04 unifactorial reference:","available" if unifactorial_reference is not None else "not available (optional)")


def save_table(frame,name):
    target=tables_dir/name
    frame.to_csv(target,index=False,float_format="%.12g")
    return target

save_table(PLAN,"planned_continuous_strata.csv")

Source: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/outputs/dataset_final.csv
Input records (unchanged): 402
Output: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_03_conditional_continuous_profiles_and_synthesis
07.01 support rows: 10 | joint profiles: 462
07.02 planned contrasts: 28 | effects: 196
06.04 unifactorial reference: available


PosixPath('/home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_03_conditional_continuous_profiles_and_synthesis/tables/planned_continuous_strata.csv')

## 2. Stratum support and observed predictor range

For each predefined factor–stratifier family, all original category levels are retained, including any empty ones. The support table records the number of source records in each stratum, valid observed predictor values, unique predictor values, and predictor quantiles in the **original units**. This supports interpretation of restricted predictor ranges, ties and sparse strata before interpreting any correlation.

In [3]:
support_rows=[]
for p in PLAN.itertuples(index=False):
    for level in FACTOR_LEVELS[p.stratifier]:
        subset=data.loc[data[p.stratifier]==level]
        xall=pd.to_numeric(subset[p.predictor],errors="coerce").to_numpy(dtype=float)
        xv=xall[np.isfinite(xall)]
        q=np.quantile(xv,[.05,.25,.5,.75,.95]) if len(xv) else [np.nan]*5
        support_rows.append({
            "pair_id":p.pair_id,"predictor":p.predictor,"predictor_label":p.predictor_label,
            "stratifier":p.stratifier,"stratifier_label":p.stratifier_label,"stratum":level,
            "n_stratum":len(subset),"n_x_valid":len(xv),"n_x_unique":len(np.unique(xv)),
            "x_minimum":float(np.min(xv)) if len(xv) else np.nan,
            "x_q05":q[0],"x_q25":q[1],"x_median":q[2],"x_q75":q[3],"x_q95":q[4],
            "x_maximum":float(np.max(xv)) if len(xv) else np.nan,
            "support_status":("empty_stratum" if len(subset)==0 else
                "small_n_below_10" if len(subset)<10 else
                "modest_n_below_20" if len(subset)<20 else "n_at_least_20"),
        })
support=pd.DataFrame(support_rows)
save_table(support,"conditional_predictor_support.csv")
display(support[["pair_id","predictor_label","stratifier_label","stratum",
                 "n_stratum","n_x_unique","support_status"]])

,pair_id,predictor_label,stratifier_label,stratum,n_stratum,n_x_unique,support_status
0,P01,Age (years),Sex,Female,294,28,n_at_least_20
1,P01,Age (years),Sex,Male,108,25,n_at_least_20
2,P02,Weight (kg),Sex,Female,294,136,n_at_least_20
3,P02,Weight (kg),Sex,Male,108,77,n_at_least_20
4,P03,Height (cm),Sex,Female,294,32,n_at_least_20
5,P03,Height (cm),Sex,Male,108,33,n_at_least_20
6,P04,BMI (kg/m²),Sex,Female,294,253,n_at_least_20
7,P04,BMI (kg/m²),Sex,Male,108,102,n_at_least_20
8,P05,SBP (mmHg),Sex,Female,294,74,n_at_least_20
9,P05,SBP (mmHg),Sex,Male,108,51,n_at_least_20


## 3. Spearman estimation and paired bootstrap

For every predefined stratum and all seven outcomes, Spearman's rank correlation ρ, its two-sided SciPy p-value and a percentile **paired-record bootstrap** 95% CI are estimated. Within each bootstrap replicate, x/y observation pairs are sampled together from the same existing stratum, retaining within-record dependence. The sample size is reported for **each outcome**; an undefined correlation is left missing when there are fewer than three valid pairs or either variable is constant. For a small stratum, interpret asymptotic p-values cautiously and emphasize the estimate and uncertainty.

The vectorized bootstrap computes the Pearson correlation of **average ranks** row by row (algebraically equivalent to SciPy Spearman, including ties), so acceleration does not change the statistical estimator. Point-estimate p-values always come from SciPy's Spearman implementation. The bootstrap replicates are cached in memory to support direct differences between disjoint two-level strata without additional resampling.

In [4]:
BOOT_RNG=np.random.default_rng(SEED)
TAIL=(1-CONFIDENCE_LEVEL)/2


def spearman_bootstrap(x,y):
    """Vectorized paired bootstrap of exact average-rank Spearman correlations.

    Row-wise ranked Pearson correlation is algebraically Spearman rho, including ties.
    Constant bootstrap replicates are undefined and kept as NaN. No approximation
    of the statistical estimator or percentile CI is introduced by vectorization.
    """
    n=len(x)
    if n<3 or len(np.unique(x))<2 or len(np.unique(y))<2:
        return np.full(N_BOOT,np.nan,dtype=float)
    idx=BOOT_RNG.integers(0,n,size=(N_BOOT,n))
    rx=stats.rankdata(x[idx],axis=1,method="average")
    ry=stats.rankdata(y[idx],axis=1,method="average")
    rx-=rx.mean(axis=1,keepdims=True)
    ry-=ry.mean(axis=1,keepdims=True)
    numerator=np.einsum("ij,ij->i",rx,ry)
    denominator=np.sqrt(np.einsum("ij,ij->i",rx,rx)*np.einsum("ij,ij->i",ry,ry))
    return np.divide(numerator,denominator,out=np.full(N_BOOT,np.nan),where=denominator>0)


def percentile_interval(draws):
    z=np.asarray(draws,dtype=float)
    z=z[np.isfinite(z)]
    if not len(z):
        return np.nan,np.nan,0
    lo,hi=np.quantile(z,[TAIL,1-TAIL])
    return float(lo),float(hi),int(len(z))


def bh_fdr(pvalues):
    ps=np.asarray(pvalues,dtype=float)
    q=np.full(ps.size,np.nan,dtype=float)
    keep=np.flatnonzero(np.isfinite(ps))
    if not len(keep):
        return q
    ordered=keep[np.argsort(ps[keep])]
    k=len(ordered)
    vals=ps[ordered]*k/np.arange(1,k+1)
    vals=np.minimum.accumulate(vals[::-1])[::-1]
    q[ordered]=np.minimum(vals,1.0)
    return q

cache={}
association_rows=[]
for p in PLAN.itertuples(index=False):
    for level in FACTOR_LEVELS[p.stratifier]:
        subset=data.loc[data[p.stratifier]==level]
        xall=pd.to_numeric(subset[p.predictor],errors="coerce").to_numpy(dtype=float)
        for outcome,label in MEASURES.items():
            yall=pd.to_numeric(subset[outcome],errors="coerce").to_numpy(dtype=float)
            valid=np.isfinite(xall)&np.isfinite(yall)
            x,y=xall[valid],yall[valid]
            n=len(x)
            family="primary_hrv" if outcome in PRIMARY else "complementary_cardiac"
            if n<3:
                rho=p_raw=np.nan
                status="undefined_fewer_than_3_pairs"
                draws=np.full(N_BOOT,np.nan)
            elif len(np.unique(x))<2 or len(np.unique(y))<2:
                rho=p_raw=np.nan
                status="undefined_constant_x_or_y"
                draws=np.full(N_BOOT,np.nan)
            else:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    rs=stats.spearmanr(x,y)
                rho,p_raw=float(rs.statistic),float(rs.pvalue)
                draws=spearman_bootstrap(x,y)
                status="estimated_small_n" if n<20 else "estimated"
            ci_lo,ci_hi,n_boot_valid=percentile_interval(draws)
            cache[(p.pair_id,level,outcome)]=draws
            association_rows.append({
                "pair_id":p.pair_id,"predictor":p.predictor,"predictor_label":p.predictor_label,
                "stratifier":p.stratifier,"stratifier_label":p.stratifier_label,"stratum":level,
                "outcome":outcome,"outcome_label":label,"family":family,
                "n_stratum":len(subset),"n_valid_pairs":n,
                "n_x_unique":len(np.unique(x)),"n_y_unique":len(np.unique(y)),
                "spearman_rho":rho,"rho_ci_low":ci_lo,"rho_ci_high":ci_hi,
                "bootstrap_valid_draws":n_boot_valid,"p_raw":p_raw,
                "estimation_status":status,
            })
conditional=pd.DataFrame(association_rows)
print("Conditional estimates planned:",len(conditional))
print("Unestimable pairs:",int(conditional.spearman_rho.isna().sum()))
display(conditional[["pair_id","stratum","outcome_label","n_valid_pairs",
                     "spearman_rho","rho_ci_low","rho_ci_high","estimation_status"]].head(18))

Conditional estimates planned: 273
Unestimable pairs: 0


,pair_id,stratum,outcome_label,n_valid_pairs,spearman_rho,rho_ci_low,rho_ci_high,estimation_status
0,P01,Female,SDNN,294,-0.038354,-0.154677,0.078152,estimated
1,P01,Female,RMSSD,294,-0.007078,-0.125653,0.108781,estimated
2,P01,Female,HF,294,-0.112880,-0.232064,0.000761,estimated
3,P01,Female,LF,294,-0.094374,-0.214325,0.018792,estimated
4,P01,Female,VLF,294,-0.104889,-0.212674,0.005845,estimated
5,P01,Female,Mean RR,294,0.174269,0.068621,0.289255,estimated
6,P01,Female,Mean HR,294,-0.173046,-0.284586,-0.063529,estimated
7,P01,Male,SDNN,108,0.329843,0.120818,0.518190,estimated
8,P01,Male,RMSSD,108,0.075949,-0.111048,0.283320,estimated
9,P01,Male,HF,108,-0.005422,-0.197215,0.192151,estimated


## 4. Multiplicity within continuous-factor families

Benjamini–Hochberg correction is applied **within each prespecified predictor × stratifier family**, separately for the five primary HRV measures and the two complementary cardiac measures. All planned *estimable* tests in that family are included, even when p is non-significant. Undefined tests remain missing, not zero. The number of valid tests used in each local family is explicitly exported.

**Important:** these are within-07.03, *within-pair-family* q-values. They do **not** control FDR across all 17 families, across 07.02, across stage 06, or study-wide. Stage 06 and 07.02 retain their independent documented families; no pooled correction is retroactively applied here.

In [5]:
conditional["p_fdr_bh"]=np.nan
family_rows=[]
for p in PLAN.itertuples(index=False):
    for family in ["primary_hrv","complementary_cardiac"]:
        mask=(conditional.pair_id==p.pair_id)&(conditional.family==family)
        idx=conditional.index[mask]
        conditional.loc[idx,"p_fdr_bh"]=bh_fdr(conditional.loc[idx,"p_raw"].to_numpy(dtype=float))
        part=conditional.loc[idx]
        family_rows.append({
            "pair_id":p.pair_id,"predictor":p.predictor,"stratifier":p.stratifier,
            "family":family,"n_planned_tests":len(part),
            "n_estimable_tests":int(part.p_raw.notna().sum()),
            "n_undefined_tests":int(part.p_raw.isna().sum()),
            "n_within_family_q_below_0_05":int((part.p_fdr_bh<.05).sum()),
        })
family_summary=pd.DataFrame(family_rows)
save_table(conditional,"conditional_spearman_associations.csv")
save_table(family_summary,"conditional_fdr_family_summary.csv")
display(family_summary)

,pair_id,predictor,stratifier,family,n_planned_tests,n_estimable_tests,n_undefined_tests,n_within_family_q_below_0_05
0,P01,age,sex_category,primary_hrv,10,10,0,1
1,P01,age,sex_category,complementary_cardiac,4,4,0,2
2,P02,weight_kg,sex_category,primary_hrv,10,10,0,0
3,P02,weight_kg,sex_category,complementary_cardiac,4,4,0,0
4,P03,height_cm,sex_category,primary_hrv,10,10,0,1
5,P03,height_cm,sex_category,complementary_cardiac,4,4,0,0
6,P04,imc,sex_category,primary_hrv,10,10,0,0
7,P04,imc,sex_category,complementary_cardiac,4,4,0,0
8,P05,bp_systolic,sex_category,primary_hrv,10,10,0,0
9,P05,bp_systolic,sex_category,complementary_cardiac,4,4,0,0


## 5. Direct estimation of differences in stratum correlations

For the twelve families whose existing stratifier has **exactly two levels**, estimate

`Δρ = ρ(comparison stratum) − ρ(reference stratum)`.

The direction is **Female − Male**, **SBP ≥140 − SBP <140**, or **DBP ≥90 − DBP <90**, according to the original stratifier. The percentile bootstrap resamples the two disjoint strata independently; for each outcome, the two resampled correlations are subtracted. This directly quantifies uncertainty of heterogeneity in rank association, instead of comparing p-values.

These are **exploratory difference-of-correlations descriptors**. No p-values or FDR are assigned to Δρ, and they are not a formal model-based interaction or covariate-adjusted test. In particular, different Spearman correlations can reflect different observed predictor ranges, ties, sample composition or nonlinearity. The original source groups are preserved.

In [6]:
BINARY_DIRECTIONS={
    "sex_category":("Female","Male"),
    "sbp_category":(">=140 mmHg","<140 mmHg"),
    "dbp_category":(">=90 mmHg","<90 mmHg"),
}
difference_rows=[]
for p in PLAN.itertuples(index=False):
    if p.stratifier not in BINARY_DIRECTIONS:
        continue
    comparison,reference=BINARY_DIRECTIONS[p.stratifier]
    for outcome,label in MEASURES.items():
        a=conditional.loc[(conditional.pair_id==p.pair_id)&(conditional.stratum==comparison)&
                          (conditional.outcome==outcome)].iloc[0]
        b=conditional.loc[(conditional.pair_id==p.pair_id)&(conditional.stratum==reference)&
                          (conditional.outcome==outcome)].iloc[0]
        da,db=cache[(p.pair_id,comparison,outcome)],cache[(p.pair_id,reference,outcome)]
        valid_draws=da-db
        dlo,dhi,nb=percentile_interval(valid_draws)
        d=(float(a.spearman_rho-b.spearman_rho) if pd.notna(a.spearman_rho) and
            pd.notna(b.spearman_rho) else np.nan)
        difference_rows.append({
            "pair_id":p.pair_id,"predictor":p.predictor,"predictor_label":p.predictor_label,
            "stratifier":p.stratifier,"comparison_stratum":comparison,
            "reference_stratum":reference,"outcome":outcome,"outcome_label":label,
            "family":"primary_hrv" if outcome in PRIMARY else "complementary_cardiac",
            "n_comparison_valid":int(a.n_valid_pairs),"n_reference_valid":int(b.n_valid_pairs),
            "rho_comparison":a.spearman_rho,"rho_reference":b.spearman_rho,
            "delta_rho_comparison_minus_reference":d,
            "delta_ci_low":dlo,"delta_ci_high":dhi,
            "bootstrap_valid_draws":nb,
            "estimation_status":"estimated_descriptive" if np.isfinite(d) else "not_estimable",
        })
differences=pd.DataFrame(difference_rows)
save_table(differences,"conditional_rho_differences_descriptive.csv")
print("Exploratory difference-of-rho rows:",len(differences))
display(differences[["pair_id","comparison_stratum","reference_stratum","outcome_label",
                     "delta_rho_comparison_minus_reference","delta_ci_low","delta_ci_high"]].head(14))

Exploratory difference-of-rho rows: 84


,pair_id,comparison_stratum,reference_stratum,outcome_label,delta_rho_comparison_minus_reference,delta_ci_low,delta_ci_high
0,P01,Female,Male,SDNN,-0.368197,-0.583264,-0.140519
1,P01,Female,Male,RMSSD,-0.083028,-0.307505,0.143722
2,P01,Female,Male,HF,-0.107458,-0.335037,0.109817
3,P01,Female,Male,LF,-0.067312,-0.307967,0.162231
4,P01,Female,Male,VLF,-0.165664,-0.384298,0.057317
5,P01,Female,Male,Mean RR,0.013152,-0.193070,0.232893
6,P01,Female,Male,Mean HR,-0.014793,-0.223938,0.197893
7,P02,Female,Male,SDNN,-0.122073,-0.337710,0.095787
8,P02,Female,Male,RMSSD,0.074400,-0.151491,0.294485
9,P02,Female,Male,HF,0.169200,-0.055648,0.382098


## 6. Cross-stage synthesis without repeating upstream tests

The synthesis uses existing CSV outputs from **07.01** (ten categorical pair grids and all seven-outcome descriptive profiles) and **07.02** (prespecified categorical effect estimates and FDR bookkeeping), plus the new 07.03 association and Δρ tables.

The consolidated inventory describes **coverage and estimation status**, not a competition for the smallest p-value. Multiple profiles reuse the same 402 records and are not independent replicates. The optional stage-06.04 Spearman results are joined solely as an overlapping **unifactorial reference**, without a new test or CI for the difference between overlapping estimates.

In [7]:
up=source_tables
n_cells=int(len(up["cell_counts"]))
n_occupied=int(pd.to_numeric(up["cell_counts"]["n"],errors="coerce").gt(0).sum())
n_targeted=len(up["targeted_effects"])
n_targeted_est=int(up["targeted_effects"]["p_raw"].notna().sum())

inventory=pd.DataFrame([
    {"stage":"07.01", "component":"joint categorical pairs", "planned":int(len(up["cell_support"])),
     "estimated_or_occupied":int(len(up["cell_support"])),"not_estimable":0,
     "interpretation":"all original categorical pairings"},
    {"stage":"07.01", "component":"joint cells", "planned":n_cells,
     "estimated_or_occupied":n_occupied,"not_estimable":n_cells-n_occupied,
     "interpretation":"empty cells retained; not inferential"},
    {"stage":"07.01", "component":"cell × outcome descriptive profiles",
     "planned":len(up["joint_profiles"]),
     "estimated_or_occupied":int(pd.to_numeric(up["joint_profiles"]["n"],errors="coerce").gt(0).sum()),
     "not_estimable":int(pd.to_numeric(up["joint_profiles"]["n"],errors="coerce").eq(0).sum()),
     "interpretation":"descriptive only; finite-sample quantiles"},
    {"stage":"07.02", "component":"prespecified categorical contrasts",
     "planned":len(up["planned_contrasts"]),
     "estimated_or_occupied":int(up["planned_contrasts"]["estimable_cell_pair"].astype(str).str.lower().eq("true").sum()),
     "not_estimable":int(up["planned_contrasts"]["estimable_cell_pair"].astype(str).str.lower().eq("false").sum()),
     "interpretation":"fixed contrast ledger; no formal interaction model"},
    {"stage":"07.02", "component":"categorical contrast × outcome estimates",
     "planned":n_targeted,"estimated_or_occupied":n_targeted_est,
     "not_estimable":n_targeted-n_targeted_est,
     "interpretation":"Cliff delta, median difference, MW U and local BH q"},
    {"stage":"07.03", "component":"continuous-factor × categorical-stratifier families",
     "planned":len(PLAN),"estimated_or_occupied":len(PLAN),"not_estimable":0,
     "interpretation":"predefined conditional continuous questions"},
    {"stage":"07.03", "component":"strata × outcome associations",
     "planned":len(conditional),"estimated_or_occupied":int(conditional.spearman_rho.notna().sum()),
     "not_estimable":int(conditional.spearman_rho.isna().sum()),
     "interpretation":"Spearman rho, paired percentile CI, local BH q"},
    {"stage":"07.03", "component":"binary-stratum direct delta-rho descriptors",
     "planned":len(differences),
     "estimated_or_occupied":int(differences.delta_rho_comparison_minus_reference.notna().sum()),
     "not_estimable":int(differences.delta_rho_comparison_minus_reference.isna().sum()),
     "interpretation":"estimation-only difference of correlations; no p/q"},
])
save_table(inventory,"phase_07_coverage_inventory.csv")

outcome_coverage=[]
for outcome,label in MEASURES.items():
    o01=up["joint_profiles"].loc[up["joint_profiles"].outcome==outcome]
    o02=up["targeted_effects"].loc[up["targeted_effects"].outcome==outcome]
    o03=conditional.loc[conditional.outcome==outcome]
    od=differences.loc[differences.outcome==outcome]
    outcome_coverage.append({
        "outcome":outcome,"outcome_label":label,
        "outcome_class":"primary_hrv" if outcome in PRIMARY else "complementary_cardiac",
        "joint_profiles_planned":len(o01),
        "joint_profiles_occupied":int(pd.to_numeric(o01.n,errors="coerce").gt(0).sum()),
        "categorical_contrasts_planned":len(o02),
        "categorical_contrasts_estimated":int(o02.p_raw.notna().sum()),
        "conditional_associations_planned":len(o03),
        "conditional_associations_estimated":int(o03.spearman_rho.notna().sum()),
        "direct_delta_rho_planned":len(od),
        "direct_delta_rho_estimated":int(od.delta_rho_comparison_minus_reference.notna().sum()),
    })
coverage=pd.DataFrame(outcome_coverage)
save_table(coverage,"phase_07_outcome_coverage.csv")

# Tidy cross-stage effect registry: preserve estimator, units, direction, original
# source-specific p/q and uncertainty. Different estimand scales are NOT combined.
registry_rows=[]
for r in up["targeted_effects"].itertuples(index=False):
    descriptor=f"{r.comparison_a} / {r.comparison_b} versus {r.reference_a} / {r.reference_b}"
    common={"source_stage":"07.02", "source_id":r.contrast_id, "question":r.contrast_label,
            "predictor":r.factor_a,"stratifier":r.factor_b,"comparison":descriptor,
            "reference":"", "outcome":r.outcome,"outcome_label":r.outcome_label,
            "outcome_class":r.family,"n_comparison":r.n_comparison_valid,
            "n_reference":r.n_reference_valid}
    registry_rows.append({**common,"effect_type":"cliffs_delta","effect_scale":"[-1, 1]",
        "effect_value":r.cliffs_delta_comparison_vs_reference,
        "ci_low":r.cliffs_ci_low,"ci_high":r.cliffs_ci_high,
        "p_raw":r.p_raw,"p_fdr_bh":r.p_fdr_bh,
        "inferential_scope":"Mann-Whitney rank-distribution comparison; local 07.02 BH"})
    registry_rows.append({**common,"effect_type":"median_difference","effect_scale":"original outcome units",
        "effect_value":r.median_difference_comparison_minus_reference,
        "ci_low":r.median_diff_ci_low,"ci_high":r.median_diff_ci_high,
        "p_raw":np.nan,"p_fdr_bh":np.nan,
        "inferential_scope":"Descriptive median difference; 07.02 Mann-Whitney p/q is not a median test"})
for r in conditional.itertuples(index=False):
    registry_rows.append({
        "source_stage":"07.03", "source_id":r.pair_id,
        "question":f"{r.predictor_label} within {r.stratifier_label}: {r.stratum}",
        "predictor":r.predictor,"stratifier":r.stratifier,"comparison":r.stratum,"reference":"",
        "outcome":r.outcome,"outcome_label":r.outcome_label,"outcome_class":r.family,
        "n_comparison":r.n_valid_pairs,"n_reference":np.nan,
        "effect_type":"spearman_rho","effect_scale":"[-1, 1]",
        "effect_value":r.spearman_rho,"ci_low":r.rho_ci_low,"ci_high":r.rho_ci_high,
        "p_raw":r.p_raw,"p_fdr_bh":r.p_fdr_bh,
        "inferential_scope":"Within-stratum asymptotic Spearman p; local 07.03 BH"})
for r in differences.itertuples(index=False):
    registry_rows.append({
        "source_stage":"07.03", "source_id":r.pair_id,
        "question":f"Difference in {r.predictor_label}–HRV correlation",
        "predictor":r.predictor,"stratifier":r.stratifier,
        "comparison":r.comparison_stratum,"reference":r.reference_stratum,
        "outcome":r.outcome,"outcome_label":r.outcome_label,"outcome_class":r.family,
        "n_comparison":r.n_comparison_valid,"n_reference":r.n_reference_valid,
        "effect_type":"delta_spearman_rho","effect_scale":"[-2, 2]",
        "effect_value":r.delta_rho_comparison_minus_reference,
        "ci_low":r.delta_ci_low,"ci_high":r.delta_ci_high,
        "p_raw":np.nan,"p_fdr_bh":np.nan,
        "inferential_scope":"Exploratory difference-of-correlations CI; no interaction test or p/q"})
registry=pd.DataFrame(registry_rows)
save_table(registry,"phase_07_all_effects_long.csv")
registry_summary=(registry.groupby(["source_stage","effect_type"],dropna=False)
                  .agg(n_rows=("outcome","size"),n_estimable=("effect_value","count"))
                  .reset_index())
print("Cross-stage effect registry (non-interchangeable estimands):")
display(registry_summary)

if unifactorial_reference is not None:
    base=unifactorial_reference[["factor","outcome","spearman_rho","rho_ci_low","rho_ci_high",
                                  "p_raw","p_fdr_bh"]].rename(columns={
        "factor":"predictor","spearman_rho":"unifactorial_rho",
        "rho_ci_low":"unifactorial_ci_low","rho_ci_high":"unifactorial_ci_high",
        "p_raw":"unifactorial_p_raw","p_fdr_bh":"unifactorial_domain_q"})
    comparison=conditional.merge(base,on=["predictor","outcome"],how="left",validate="many_to_one")
    comparison["rho_conditional_minus_unifactorial"]=(
        comparison.spearman_rho-comparison.unifactorial_rho)
    save_table(comparison,"conditional_vs_unifactorial_descriptive_reference.csv")
else:
    comparison=None

print("Phase 07 synthesis inventory:")
display(inventory)
display(coverage)

Cross-stage effect registry (non-interchangeable estimands):


,source_stage,effect_type,n_rows,n_estimable
0,07.02,cliffs_delta,196,189
1,07.02,median_difference,196,189
2,07.03,delta_spearman_rho,84,84
3,07.03,spearman_rho,273,273


Phase 07 synthesis inventory:


,stage,component,planned,estimated_or_occupied,not_estimable,interpretation
0,07.01,joint categorical pairs,10,10,0,all original categorical pairings
1,07.01,joint cells,66,64,2,empty cells retained; not inferential
2,07.01,cell × outcome descriptive profiles,462,448,14,descriptive only; finite-sample quantiles
3,07.02,prespecified categorical contrasts,28,27,1,fixed contrast ledger; no formal interaction m...
4,07.02,categorical contrast × outcome estimates,196,189,7,"Cliff delta, median difference, MW U and local..."
5,07.03,continuous-factor × categorical-stratifier fam...,17,17,0,predefined conditional continuous questions
6,07.03,strata × outcome associations,273,273,0,"Spearman rho, paired percentile CI, local BH q"
7,07.03,binary-stratum direct delta-rho descriptors,84,84,0,estimation-only difference of correlations; no...


,outcome,outcome_label,outcome_class,joint_profiles_planned,joint_profiles_occupied,categorical_contrasts_planned,categorical_contrasts_estimated,conditional_associations_planned,conditional_associations_estimated,direct_delta_rho_planned,direct_delta_rho_estimated
0,t2m_pre_sdnn,SDNN,primary_hrv,66,64,28,27,39,39,12,12
1,t2m_pre_rmssd,RMSSD,primary_hrv,66,64,28,27,39,39,12,12
2,t2m_pre_hf,HF,primary_hrv,66,64,28,27,39,39,12,12
3,t2m_pre_lf,LF,primary_hrv,66,64,28,27,39,39,12,12
4,t2m_pre_vlf,VLF,primary_hrv,66,64,28,27,39,39,12,12
5,t2m_pre_mean_rr,Mean RR,complementary_cardiac,66,64,28,27,39,39,12,12
6,t2m_pre_mean_hr,Mean HR,complementary_cardiac,66,64,28,27,39,39,12,12


## 7. Figure conventions

Figures separate overview and detailed biological interpretation:

1. **One coverage chart** displays original stratum support, independent of inferential results.
2. **Seventeen ρ heatmaps** show all outcomes and all existing levels for each continuous × categorical pair. The colorbar is a common signed Spearman scale; the five primary and two complementary cardiac measures are visibly separated.
3. **Seventeen all-outcome scatter atlases** show raw source observations, colored by existing strata. LOWESS lines are **visual descriptions only** and are suppressed for small or low-variability groups; no trend-model p-values are computed. HF, LF and VLF are `log1p` only for these figures, with labels stating so.
4. **Two Δρ forest sheets** show the exploratory direct differences in stratum-specific correlations and percentile-bootstrap intervals, separately for primary and complementary outcomes.

Each panel reserves explicit space for a figure title, subtitle, and legends; small/empty strata and undefined estimates remain visibly identified. All statistical estimation continues to use the original untransformed measurements.

In [8]:
figure_manifest=[]


def write_figure(fig,filename,role,description,pair_id="all"):
    target=figure_dir/filename
    fig.savefig(target,dpi=PLOT_DPI,bbox_inches="tight",facecolor="white")
    if SHOW_FIGURES:
        display(fig)
    plt.close(fig)
    figure_manifest.append({"figure":str(target.relative_to(output_root)),
                            "pair_id":pair_id,"role":role,"description":description})


def add_header(fig,title,subtitle):
    # Use physical-inch offsets so compact heatmaps get the same readable header clearance
    # as the larger scatter atlases and forest sheets.
    height=fig.get_figheight()
    fig.suptitle(title,x=.06,y=1-.18/height,ha="left",fontsize=17,fontweight="bold",color=NAVY)
    fig.text(.06,1-.67/height,subtitle,ha="left",va="top",fontsize=9.7,color=MUTED)


def format_stratum(s):
    return str(s).replace(">=","≥")


def plot_support_overview():
    fig,ax=plt.subplots(figsize=(13,9.0))
    add_header(fig,"Coverage of conditional continuous analyses",
               "Original cohort support per predefined continuous predictor × stratum | every original level shown")
    fig.subplots_adjust(left=.30,right=.95,top=.875,bottom=.07)
    labels=[];counts=[];colors=[];ys=[]
    pos=0
    for p in PLAN.itertuples(index=False):
        part=support.loc[support.pair_id==p.pair_id]
        for idx,r in enumerate(part.itertuples(index=False)):
            labels.append(f"{p.predictor_label}  |  {format_stratum(r.stratum)}")
            counts.append(r.n_stratum)
            colors.append(STRATUM_COLORS[idx])
            ys.append(pos);pos+=1
        pos+=.45
    ax.barh(ys,counts,color=colors,alpha=.86,height=.75)
    for y,count in zip(ys,counts):
        ax.text(count+2,y,str(count),va="center",fontsize=8,color=NAVY)
    ax.set_yticks(ys,labels,fontsize=7.6)
    ax.invert_yaxis();ax.set_xlim(0,max(counts)*1.13)
    ax.set_xlabel("Source records within the original stratum")
    ax.grid(axis="x",alpha=.55);ax.grid(axis="y",visible=False)
    ax.set_title("17 planned predictor–stratifier questions | 39 original strata",loc="left",pad=10,fontsize=11)
    write_figure(fig,"00_continuous_stratum_coverage.png","support_overview",
                 "Actual support per existing stratum; all records retained")


def plot_rho_heatmap(p):
    sub=conditional.loc[conditional.pair_id==p.pair_id]
    levels=FACTOR_LEVELS[p.stratifier]
    mat=sub.pivot(index="stratum",columns="outcome",values="spearman_rho").reindex(
        index=levels,columns=OUTCOME_KEYS)
    fig,ax=plt.subplots(figsize=(12.8,max(4.0,2.4+.62*len(levels))))
    add_header(fig,f"{p.predictor_label} within {p.stratifier_label.lower()}",
               "Spearman ρ in original scale | paired bootstrap 95% CIs and local BH q are exported to CSV")
    fig.subplots_adjust(left=.15,right=.87,top=.67,bottom=.16)
    sns.heatmap(mat,ax=ax,vmin=-1,vmax=1,center=0,cmap="vlag",linewidths=1,
                linecolor="white",annot=False,cbar_kws={"label":"Spearman ρ", "shrink":.72})
    ax.set_xlabel("Cardiac measurement");ax.set_ylabel("")
    ax.set_xticklabels(list(MEASURES.values()),rotation=0,fontsize=9)
    ax.set_yticklabels([format_stratum(l) for l in levels],rotation=0,fontsize=9)
    ax.axvline(5,color=NAVY,lw=2.2,alpha=.8)
    for j,lev in enumerate(levels):
        for i,col in enumerate(OUTCOME_KEYS):
            r=mat.loc[lev,col]
            if np.isfinite(r):
                ax.text(i+.5,j+.5,f"{r:+.2f}",ha="center",va="center",fontsize=9,
                        color="white" if abs(r)>.55 else NAVY,weight="medium")
            else:
                ax.text(i+.5,j+.5,"N/E",ha="center",va="center",fontsize=9,color=NAVY)
    ax.text(2.5,-.14,"Primary HRV",ha="center",va="center",fontsize=9.5,color=MUTED)
    ax.text(6,-.14,"Complementary",ha="center",va="center",fontsize=9.5,color=MUTED)
    write_figure(fig,f"{p.pair_id.lower()}_conditional_rho_heatmap.png","rho_heatmap",
                 f"Seven outcome correlations in original {p.stratifier_label.lower()} levels",p.pair_id)


def plot_scatter_atlas(p):
    subset_names=FACTOR_LEVELS[p.stratifier]
    fig=plt.figure(figsize=(16.5,11.8))
    add_header(fig,f"{p.predictor_label} × cardiac measures by {p.stratifier_label.lower()}",
               "Raw observations | descriptive LOWESS where support permits | spectral variables use visual-only log1p when defined")
    gs=fig.add_gridspec(3,3,left=.075,right=.965,top=.815,bottom=.065,hspace=.50,wspace=.31)
    legend=[Line2D([0],[0],marker="o",color=STRATUM_COLORS[k],markerfacecolor=STRATUM_COLORS[k],
                   lw=1.3,markersize=7,label=f"{format_stratum(l)} (n={len(data[data[p.stratifier]==l])})")
            for k,l in enumerate(subset_names)]
    fig.legend(handles=legend,loc="upper left",bbox_to_anchor=(.075,.885),frameon=False,
               ncol=min(4,len(legend)),fontsize=9)
    for i,(outcome,label) in enumerate(MEASURES.items()):
        ax=fig.add_subplot(gs[i//3,i%3])
        source_values=pd.to_numeric(data[outcome],errors="coerce").to_numpy(dtype=float)
        visual_log=(outcome in ("t2m_pre_hf","t2m_pre_lf","t2m_pre_vlf") and
                    np.all(source_values[np.isfinite(source_values)]>=0))
        for k,lev in enumerate(subset_names):
            raw=data.loc[data[p.stratifier]==lev]
            xraw=pd.to_numeric(raw[p.predictor],errors="coerce").to_numpy(dtype=float)
            yraw=pd.to_numeric(raw[outcome],errors="coerce").to_numpy(dtype=float)
            valid=np.isfinite(xraw)&np.isfinite(yraw)
            xx,yy=xraw[valid],yraw[valid]
            if visual_log:
                yy=np.log1p(yy)
            if not len(xx):
                continue
            ax.scatter(xx,yy,s=13,alpha=.44,color=STRATUM_COLORS[k],edgecolors="none",rasterized=True)
            if LOWESS_AVAILABLE and len(xx)>=25 and len(np.unique(xx))>=8:
                try:
                    curve=lowess(yy,xx,frac=.55,it=0,return_sorted=True)
                    ax.plot(curve[:,0],curve[:,1],color=STRATUM_COLORS[k],lw=2.0,alpha=.95)
                except (ValueError,RuntimeWarning,FloatingPointError):
                    pass
        ax.set_title(label+(" | visual log1p" if visual_log else ""),loc="left",pad=10,fontsize=10)
        ax.set_xlabel(p.predictor_label,fontsize=9)
        ax.set_ylabel("log1p(source)" if visual_log else label,fontsize=9)
        ax.tick_params(labelsize=8);ax.grid(alpha=.38)
    for i in range(len(MEASURES),9):
        ax=fig.add_subplot(gs[i//3,i%3]);ax.axis("off")
    write_figure(fig,f"{p.pair_id.lower()}_all_outcome_scatter_atlas.png","scatter_atlas",
                 f"Observed continuous relationship and visual-only LOWESS by {p.stratifier_label.lower()}",p.pair_id)


def plot_rho_delta_forest(outcomes,filename,title):
    rows=[p for p in PLAN.itertuples(index=False) if p.stratifier in BINARY_DIRECTIONS]
    ncol=len(outcomes);nrows=len(rows)
    fig=plt.figure(figsize=(16.5 if ncol==5 else 10.4,8.5))
    add_header(fig,title,
               "Δρ = comparison stratum − reference stratum | independent-stratum bootstrap 95% CI | descriptive, no p/q")
    gs=fig.add_gridspec(1,ncol,left=.26 if ncol==5 else .39,right=.975,top=.83,bottom=.105,
                        wspace=.26)
    bound=np.nanmax(np.abs(differences[["delta_ci_low","delta_ci_high"]].to_numpy(dtype=float)))
    bound=max(.35,min(1.8,bound*1.12)) if np.isfinite(bound) else .35
    for i,(outcome,label) in enumerate(outcomes.items()):
        ax=fig.add_subplot(gs[0,i]);ys=np.arange(nrows)
        ax.axvline(0,color=NAVY,lw=1,alpha=.8)
        for k,p in enumerate(rows):
            item=differences.loc[(differences.pair_id==p.pair_id)&(differences.outcome==outcome)]
            if item.empty: continue
            r=item.iloc[0]
            if np.isfinite(r.delta_rho_comparison_minus_reference):
                v=r.delta_rho_comparison_minus_reference
                if np.isfinite(r.delta_ci_low) and np.isfinite(r.delta_ci_high):
                    ax.plot([r.delta_ci_low,r.delta_ci_high],[k,k],color=TEAL,lw=2,solid_capstyle="round")
                ax.plot(v,k,marker="o",color=CORAL,markersize=5)
            else: ax.text(0,k,"N/E",fontsize=7,va="center",ha="center")
        ax.set_ylim(nrows-.5,-.5);ax.set_xlim(-bound,bound)
        ax.set_title(label,fontsize=10,pad=10)
        ax.set_yticks(ys)
        if i==0:
            ax.set_yticklabels([f"{p.predictor_label.split(' (')[0]} | {p.stratifier_label}" for p in rows],fontsize=8)
        else: ax.set_yticklabels([])
        ax.set_xlabel("Difference in Spearman ρ",fontsize=8)
        ax.grid(axis="x",alpha=.4);ax.grid(axis="y",visible=False)
    write_figure(fig,filename,"delta_rho_forest",
                 "Direct exploratory differences between correlations for binary strata")

plot_support_overview()
for p in PLAN.itertuples(index=False):
    plot_rho_heatmap(p)
    plot_scatter_atlas(p)
plot_rho_delta_forest(PRIMARY,"18_direct_rho_differences_primary_forest.png",
                     "Exploratory between-stratum differences in primary HRV correlations")
plot_rho_delta_forest(COMPLEMENTARY,"19_direct_rho_differences_cardiac_forest.png",
                     "Exploratory between-stratum differences in complementary correlations")

manifest=pd.DataFrame(figure_manifest)
save_table(manifest,"figure_manifest.csv")
print("Figures generated:",len(manifest))
display(manifest.tail(5))

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.
findfont: Failed to find font weight medium for DejaVu Sans, now using 400.
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Figures generated: 37


,figure,pair_id,role,description
32,figures/multipanels/p16_all_outcome_scatter_at...,P16,scatter_atlas,Observed continuous relationship and visual-on...
33,figures/multipanels/p17_conditional_rho_heatma...,P17,rho_heatmap,Seven outcome correlations in original age gro...
34,figures/multipanels/p17_all_outcome_scatter_at...,P17,scatter_atlas,Observed continuous relationship and visual-on...
35,figures/multipanels/18_direct_rho_differences_...,all,delta_rho_forest,Direct exploratory differences between correla...
36,figures/multipanels/19_direct_rho_differences_...,all,delta_rho_forest,Direct exploratory differences between correla...


## 8. Provenance, full result report and methodological README

These artifacts provide an auditable scientific handoff. The Markdown report displays every correlation and direct difference estimate (including non-significant and undefined entries), preserving numeric precision in CSV. p- and q-values smaller than 0.001 are rendered as inequalities in Markdown, never rounded to `0.000`. No statistical estimate is altered for presentation.

In [9]:
def fmt(v,digits=3,probability=None):
    if pd.isna(v):return "—"
    try:
        v=float(v)
    except (ValueError,TypeError):return str(v)
    if probability is not None and 0<=v<.001:
        return f"{probability} < 0.001"
    return f"{v:.{digits}f}"


def markdown_table(frame,columns,labels=None,integer_cols=()):
    labels=labels or columns
    lines=["| "+" | ".join(labels)+" |","| "+" | ".join(["---"]*len(columns))+" |"]
    for r in frame[columns].itertuples(index=False,name=None):
        vals=[]
        for col,v in zip(columns,r):
            if col in integer_cols:
                vals.append(str(int(v)) if pd.notna(v) else "—")
            elif col=="p_raw": vals.append(fmt(v,probability="p"))
            elif col=="p_fdr_bh": vals.append(fmt(v,probability="q"))
            elif isinstance(v,(float,np.floating)): vals.append(fmt(v))
            else: vals.append(str(v).replace("|","/") if pd.notna(v) else "—")
        lines.append("| "+" | ".join(vals)+" |")
    return "\n".join(lines)

run_info=pd.DataFrame([{
    "source_file":str(input_file),"n_records":len(data),"analysis_root":str(analysis_root),
    "planned_families":len(PLAN),"planned_strata":len(support),
    "planned_conditional_associations":len(conditional),
    "estimable_conditional_associations":int(conditional.spearman_rho.notna().sum()),
    "planned_binary_stratum_delta_rho":len(differences),
    "estimable_binary_stratum_delta_rho":int(differences.delta_rho_comparison_minus_reference.notna().sum()),
    "bootstrap_replicates":N_BOOT,"confidence_level":CONFIDENCE_LEVEL,"seed":SEED,
    "multiplicity":"BH within each predictor × stratifier family, separately primary vs complementary",
    "delta_rho_inference":"descriptive bootstrap CI only; no p or q",
    "source_07_01":str(stage01),"source_07_02":str(stage02),
    "source_06_04_optional":str(unifactorial_path) if unifactorial_reference is not None else "not_available",
    "figures":len(manifest),"lowess_available":LOWESS_AVAILABLE,
    "python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,
    "scipy":__import__("scipy").__version__,"seaborn":sns.__version__,
    "matplotlib":matplotlib.__version__,
}])
save_table(run_info,"run_information.csv")

result_lines=[
    "# 07.03 Conditional continuous profiles and phase 07 synthesis — Results",
    "",
    f"**Accepted input records:** {len(data)} | **Prespecified families:** {len(PLAN)} | "
    f"**Stratum–outcome estimates:** {len(conditional)} | "
    f"**Direct delta-rho descriptors:** {len(differences)} | "
    f"**Bootstrap:** {N_BOOT} draws, 95% percentile CI",
    "",
    "## Interpretation and multiplicity boundary",
    "The correlations are independent **within-stratum, unadjusted** observational descriptions, not adjusted model effects. "
    "Spearman p-values are two-sided; BH-FDR is applied **within each predictor × stratifier family**, "
    "separately for the five primary HRV outcomes and two complementary cardiac measures. "
    "These q-values are not adjusted across all 17 families or across stages 06, 07.01 and 07.02. "
    "Difference-of-rho results report bootstrap uncertainty only and **are not formal interaction tests**. "
    "No discovery claims are based solely on a difference between two p-values. "
    "The original outcome values and every source category are preserved.",
    "",
    "## Phase 07 coverage inventory",
    markdown_table(inventory,["stage","component","planned","estimated_or_occupied","not_estimable"],
                   integer_cols=("planned","estimated_or_occupied","not_estimable")),
    "",
    "## Cross-stage effect registry",
    "The saved `phase_07_all_effects_long.csv` contains the exact source 07.02 effects and new 07.03 "
    "effects in a tidy registry. Spearman rho, Cliff's delta, median differences and delta-rho "
    "have different estimands/scales and **must not be pooled, averaged or ranked together**. "
    "Mann–Whitney p/q are not assigned to median-difference rows; exploratory delta-rho has no p/q.",
    markdown_table(registry_summary,["source_stage","effect_type","n_rows","n_estimable"],
                   integer_cols=("n_rows","n_estimable")),
    "",
    "## Outcome-level coverage",
    markdown_table(coverage,["outcome_label","joint_profiles_occupied",
                              "categorical_contrasts_estimated","conditional_associations_estimated",
                              "direct_delta_rho_estimated"],
                   integer_cols=("joint_profiles_occupied","categorical_contrasts_estimated",
                                 "conditional_associations_estimated","direct_delta_rho_estimated")),
    "",
    "## Conditional Spearman estimates and percentile bootstrap confidence intervals",
    markdown_table(conditional,["pair_id","predictor_label","stratum","outcome_label","n_valid_pairs",
                                 "spearman_rho","rho_ci_low","rho_ci_high","p_raw","p_fdr_bh",
                                 "estimation_status"],integer_cols=("n_valid_pairs",)),
    "",
    "## Direct descriptive differences between binary-stratum correlations",
    markdown_table(differences,["pair_id","comparison_stratum","reference_stratum","outcome_label",
                                 "n_comparison_valid","n_reference_valid","delta_rho_comparison_minus_reference",
                                 "delta_ci_low","delta_ci_high","estimation_status"],
                   integer_cols=("n_comparison_valid","n_reference_valid")),
    "",
    "## Upstream results are preserved",
    "07.01 categorical cell profiles and 07.02 Mann–Whitney/Cliff's-delta estimates were "
    "read from source CSVs and were **not** recomputed or overwritten. "
    "Optional 06.04 unifactorial correlations, when present, are provided as overlapping descriptive references "
    "only; no statistically independent comparison is implied.",
    "",
    "## Figure inventory",
    markdown_table(manifest,["figure","pair_id","role","description"]),
    "",
    "## Limitations",
    "Bootstrap assumes record-level independence. Potential dependence within source projects is not modeled. "
    "Conditional associations are not adjusted for other covariates. Mathematical links between BMI, "
    "weight and height; between MAP/PP and SBP/DBP; and between Mean RR and Mean HR prevent "
    "interpreting repeated associations as independent biological discoveries. A non-significant association "
    "does not establish equivalence or physiological absence. Small or limited-range strata yield "
    "imprecise or undefined correlations. HF/LF/VLF physiological attribution depends on the measurement protocol. "
    "The observed data do not establish causal mechanisms or clinical reference intervals.",
    "",
]
results_path=output_root/"07_03_conditional_continuous_profiles_and_synthesis_results.md"
results_path.write_text("\n".join(result_lines),encoding="utf-8")

readme_lines=[
    "# 07.03 Conditional continuous HRV profiles and synthesis",
    "",
    "## Scientific scope",
    "Final notebook in phase 07 (two-factor HRV profiling). It analyzes the eight existing continuous "
    "predictors within original categorical strata and combines coverage evidence from 07.01 and 07.02. "
    "No new categories, cutoffs, exclusions, imputation, adjusted regressions or global interaction screening.",
    "",
    "## Source and execution",
    f"- Accepted dataset: `{input_file}`; records loaded: **{len(data)}** (unchanged).",
    "- Run 07.01 and 07.02 before 07.03; their exported tables are required. "
    "06.04 is an optional unifactorial descriptive reference.",
    "- Run from a project directory or set `INPUT_FILE_OVERRIDE` and, when needed, "
    "`ANALYSIS_ROOT_OVERRIDE`/`OUTPUT_ROOT_OVERRIDE`.",
    "- The notebook writes only inside its own 07.03 output directory; original upstream tables are read-only.",
    f"- Bootstrap: **{N_BOOT}** paired-record resamples per estimate, percentile **95% CI**, "
    f"random seed **{SEED}**.",
    "",
    "## Defined continuous-factor × categorical-stratifier pairs",
    markdown_table(PLAN,["pair_id","predictor_label","stratifier_label","n_strata_planned","comparison_type"],
                   integer_cols=("n_strata_planned",)),
    "",
    "## Estimands",
    "- Spearman rho, two-sided asymptotic p, paired-bootstrap percentile 95% CI within each original stratum.",
    "- Undefined when fewer than three valid observed x/y pairs, a constant predictor or a constant outcome. "
    "Sparse groups remain visible; neither records nor existing categories are removed.",
    "- Direct exploratory delta-rho for existing binary stratifiers, with an independent-stratum bootstrap "
    "percentile 95% CI. Direction: Female − Male; SBP high − low; DBP high − low. No delta-rho p/q "
    "and no formal regression interaction claim.",
    "- Original source units and ranks for all estimation. The `log1p` spectral display is visual only, "
    "and LOWESS is a descriptive drawing tool, not an inferential model.",
    "",
    "## Multiple-testing scope",
    "Benjamini–Hochberg correction is applied **within each of the 17 predictor–stratifier families**, "
    "separately across its five HRV primary outcomes and its two complementary cardiac outcomes. "
    "Every estimable planned level × outcome test in its family is included; non-estimable comparisons retain NA. "
    "There is **no pooled FDR across 07.03 pairs, across 07.02, across the three stage-06 domains, "
    "or study-wide**. 07.02 retains its six pair-specific FDR families and 06.01–06.03 retain "
    "their original within-domain families. The 07.03 synthesis never recalculates those source q-values.",
    "",
    "## Generated tables",
    "- `planned_continuous_strata.csv`: prespecified 17 analysis families.",
    "- `conditional_predictor_support.csv`: all 39 original strata, valid x values and original-unit distribution.",
    "- `conditional_spearman_associations.csv`: all 273 planned stratum–outcome estimates and uncertainty.",
    "- `conditional_fdr_family_summary.csv`: 34 local BH testing families (primary / complementary per pair).",
    "- `conditional_rho_differences_descriptive.csv`: 84 planned difference-of-rho descriptors with CIs.",
    "- `phase_07_coverage_inventory.csv`: stage 07.01, 07.02, 07.03 analysis coverage.",
    "- `phase_07_outcome_coverage.csv`: seven-outcome, cross-stage coverage summary.",
    "- `phase_07_all_effects_long.csv`: tidy 07.02/07.03 effect registry with estimator and original "
    "scale identified. The four effect types are non-interchangeable; no cross-scale pooling. "
    "Mann–Whitney p/q are not attributed to the descriptive median-difference estimand.",
    "- `conditional_vs_unifactorial_descriptive_reference.csv`: optional, only when 06.04 exists; overlapping "
    "unifactorial estimates from the existing 06.04 CSV, without testing their difference.",
    "- `figure_manifest.csv`: figure paths and descriptions.",
    "- `run_information.csv`: source, version and computational metadata.",
    "",
    "## Figures",
    "- One source-support overview; 17 Spearman heatmaps; 17 all-outcome scatter atlases; two "
    "difference-of-rho forest sheets (**37 figures**). All figures are placed in `figures/multipanels/`.",
    "- Original source observations are displayed by stratum. LOWESS requires sufficient observations "
    "and distinct predictor values; it is not used to assign significance. HF/LF/VLF are displayed "
    "with labeled visual-only log1p.",
    "",
    "## Interpretation boundaries",
    "All associations are cross-sectional, unadjusted, descriptive/exploratory. The delta-rho statistic "
    "is not a formal interaction model or an adjusted estimate. Potential project-level non-independence "
    "is not corrected by source-record bootstrap. Differences in predictor ranges and the modest size of "
    "some original categories affect precision. BMI is derived from weight/height; MAP and pulse pressure "
    "from SBP/DBP; Mean RR and Mean HR are intrinsically linked. Interpretation of HF, LF, VLF requires "
    "the source acquisition protocol. Null p-values do not demonstrate equivalence.",
    "",
    "## Actual run inventory",
    markdown_table(inventory,["stage","component","planned","estimated_or_occupied","not_estimable"],
                   integer_cols=("planned","estimated_or_occupied","not_estimable")),
    "",
    "## Runtime",
    f"Python {platform.python_version()}, pandas {pd.__version__}, NumPy {np.__version__}, "
    f"SciPy {__import__('scipy').__version__}, Seaborn {sns.__version__}, Matplotlib {matplotlib.__version__}. "
    f"LOWESS available: {LOWESS_AVAILABLE}.",
]
readme_path=output_root/"README_07_03_conditional_continuous_profiles_and_synthesis.md"
readme_path.write_text("\n".join(readme_lines)+"\n",encoding="utf-8")
print("Saved phase 07.03 results:",output_root)
print("- CSV tables:",len(list(tables_dir.glob("*.csv"))))
print("- Multipanel figures:",len(manifest))
print("- README:",readme_path.name)
print("- Results:",results_path.name)

Saved phase 07.03 results: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_03_conditional_continuous_profiles_and_synthesis
- CSV tables: 11
- Multipanel figures: 37
- README: README_07_03_conditional_continuous_profiles_and_synthesis.md
- Results: 07_03_conditional_continuous_profiles_and_synthesis_results.md


## 9. Closure of phase 07

The phase 07 result set is complete **after the real-source outputs are executed and audited**:

- **07.01** describes every possible joint categorical cell and keeps empty and sparsely occupied cells explicit.
- **07.02** compares a prespecified, scientifically interpretable subset of categorical profiles, without automatically testing every possible cell contrast.
- **07.03** characterizes continuous associations conditional on original categorical strata, estimates exploratory differences between disjoint binary strata, and consolidates the stage inventory without repeating earlier tests.

The phase remains **descriptive/exploratory**, does not imply statistical adjustment for unconditioned covariates, and does not infer a formal interaction solely from stratum-specific correlations or their p-values. Phase 07 can be used to prioritize interpretable manuscript figures and tables after reviewing actual results; a further model-based analysis is not automatically required.